In [1]:
# ============================================================
# STEP 1 — LOAD DATA & OVERVIEW
# Description:
# This script processes an EPC dataset to estimate the number of
# existing PV panels for each dwelling.
#
# Logic:
# - If "Peak Power" exists → panels = Peak Power / 0.47 (rounded)
# - If "Roof Area %" exists → panels = (percentage * ROOF_AREA) / 2
# - If Roof Area = 0% → panels = 0
#
# Output:
# A new column "EXISTING_PV_PANELS" is added and saved to a new CSV.
# ============================================================

import pandas as pd
import numpy as np
import re

# File paths
input_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_pv_update.csv"
output_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_existing_pv.csv"

# Load dataset
df = pd.read_csv(input_path)

# Quick check
print(df.columns)
print(df.head())

Index(['BUILDING_REFERENCE_NUMBER', 'OSG_REFERENCE_NUMBER', 'ADDRESS1',
       'ADDRESS2', 'ADDRESS3', 'POSTCODE', 'LATITUDE', 'LONGITUDE',
       'INSPECTION_DATE', 'TYPE_OF_ASSESSMENT',
       ...
       'ROOF_AREA', 'USABLE_ROOF_LENGTH', 'USABLE_ROOF_WIDTH',
       'PANELS_ALONG_LENGTH', 'PANELS_ALONG_WIDTH', 'POSSIBLE_PV_PANELS',
       'WINDOW_AREA_SINGLE', 'ASPECT_RATIO', 'WINDOW_H', 'WINDOW_W'],
      dtype='object', length=223)
   BUILDING_REFERENCE_NUMBER  OSG_REFERENCE_NUMBER              ADDRESS1  \
0                 1001829067           122009933.0  19 CULCREUCH AVENUE    
1                 1001951858           122010025.0             GLENVIEW    
2                 1001829071           122009868.0  22 CULCREUCH AVENUE    
3                 1234761001           122009968.0    1 MENZIES TERRACE    
4                 1001991633           122009892.0       50 MAIN STREET    

  ADDRESS2  ADDRESS3 POSTCODE   LATITUDE  LONGITUDE INSPECTION_DATE  \
0  FINTRY   GLASGOW   G63 0YB  5

In [2]:
# ============================================================
# STEP 2 — DEFINE PARSING FUNCTION (UPDATED)
# Description:
# Extracts PV information from PHOTO_SUPPLY and computes panel count.
# If no usable entry is found, returns 0 instead of NaN.
# ============================================================

def calculate_pv_panels(photo_supply, roof_area):
    if pd.isna(photo_supply):
        return 0
    
    text = str(photo_supply)
    
    # Case 1: Peak Power exists
    peak_match = re.search(r'Peak Power:\s*([0-9.]+)', text)
    if peak_match:
        peak_power = float(peak_match.group(1))
        panels = round(peak_power / 0.47)
        return panels
    
    # Case 2: Roof Area % exists
    roof_match = re.search(r'Roof Area:\s*([0-9.]+)%', text)
    if roof_match:
        percent = float(roof_match.group(1))
        
        # If explicitly 0%
        if percent == 0:
            return 0
        
        decimal = percent / 100
        panels = (decimal * roof_area) / 2
        return round(panels)
    
    # Default case: no valid PV info found
    return 0

In [3]:
# ============================================================
# STEP 3 — APPLY FUNCTION TO DATAFRAME
# Description:
# Applies the calculation across all rows to create new column.
# ============================================================

df["EXISTING_PV_PANELS"] = df.apply(
    lambda row: calculate_pv_panels(row["PHOTO_SUPPLY"], row["ROOF_AREA"]),
    axis=1
)

# Check results
print(df[["ROOF_AREA", "PHOTO_SUPPLY", "EXISTING_PV_PANELS"]].head(10))

    ROOF_AREA                                       PHOTO_SUPPLY  \
0   29.698485                                                NaN   
1   89.095454                                                NaN   
2   48.083261                                                NaN   
3   89.802561                                                NaN   
4  194.155227                                                NaN   
5   61.892510  Array: Roof Area: 0%; Connection: not applicab...   
6   65.954086  Array: Roof Area: 0%; Connection: not applicab...   
7  105.309953  Array: Roof Area: 0%; Connection: not applicab...   
8  108.187338  Array: Roof Area: 16%; Connection: connected t...   
9  137.809555  Array: Peak Power: 9.7; Orientation: North Eas...   

   EXISTING_PV_PANELS  
0                   0  
1                   0  
2                   0  
3                   0  
4                   0  
5                   0  
6                   0  
7                   0  
8                   9  
9          

In [4]:
# ============================================================
# STEP 4 — VALIDATION CHECKS
# Description:
# Basic sanity checks to ensure logic worked correctly.
# ============================================================

print("Summary statistics:")
print(df["EXISTING_PV_PANELS"].describe())

print("\nUnique values:")
print(sorted(df["EXISTING_PV_PANELS"].dropna().unique()))

Summary statistics:
count    117.000000
mean       1.632479
std        4.875340
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max       26.000000
Name: EXISTING_PV_PANELS, dtype: float64

Unique values:
[np.int64(0), np.int64(6), np.int64(7), np.int64(9), np.int64(11), np.int64(14), np.int64(17), np.int64(18), np.int64(21), np.int64(26)]


In [5]:
# ============================================================
# STEP 5 — SAVE OUTPUT FILE
# Description:
# Writes updated dataset to new CSV file.
# ============================================================

df.to_csv(output_path, index=False)

print(f"File saved successfully to:\n{output_path}")

File saved successfully to:
/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_existing_pv.csv
